# Configuring Dataset .ini File for Refined Cliff Dist <br>
This script is to prepare the .ini file for simulating more data to find the cliff distances for the training and testing dataset at higher resolution of 1 m.

### Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import pickle

def find_dip_after_cliff_hdist(df, threshold):
    """
    Identify the first hdist point after the 'cliff' point in the data where the value drops significantly.
    The cliff is defined as the horizontal distance right before the reliability first drops below the threshold.
    The "dip after cliff" is defined as the first horizontal distance where the reliability drops below the threshold after the cliff.

    Parameters:
    df (pd.DataFrame): DataFrame for a combination of height, USI, and Bitrate. The columns are 'Horizontal_Distance' and 'Reliability'.
    threshold (float): The reliability threshold (e.g., 0.99).

    Returns:
    hdist: The horizontal distance of the cliff point (where reliability first drops below threshold).
           Returns 0 if no cliff is found or all values are below threshold.
    """
    # Sort by horizontal distance to find the first point where reliability drops below threshold
    df_sorted = df.sort_values('Horizontal_Distance').reset_index(drop=True)
    
    if df_sorted.empty:
        return 0
    
    # Find the first index where reliability drops below threshold
    below_threshold = df_sorted[df_sorted['Reliability'] < threshold]
    
    if below_threshold.empty:
        # No point drops below threshold, return the max horizontal distance
        return df_sorted['Horizontal_Distance'].max()
    
    # Return the horizontal distance of the first point below threshold
    return below_threshold['Horizontal_Distance'].iloc[0]

## Find Cliff Dist for Training Dataset

In [2]:
'''
The cliff distance for each link may be different. Let's refine data points for all links.
Get the cliff distances for each link for each combination of MCS, USI, Height. Concatenate them and remove duplicated points.
'''

LINKS = ["Downlink", "Uplink", "Video"]
USI = [100]
BITRATE = [6.5, 13, 19.5, 26, 39, 52, 58.5, 65]  # treat as MCS
TRAIN_HEIGHT = [60, 75, 90, 105, 120, 135, 150, 165, 180, 195, 210, 225, 240, 255, 270, 285, 300]
THRESHOLD = 0.99
mcs_bitrate_map = {6.5: "BPSK", 13: "QPSK", 19.5: "QPSK", 26: "QAM-16", 39: "QAM-16", 52: "QAM-64", 58.5: "QAM-64", 65: "QAM-64"}

def add_reliability(df: pd.DataFrame) -> pd.DataFrame:
    if "Num_Sent" in df.columns and "Num_Reliable" in df.columns:
        df["Reliability"] = df["Num_Reliable"] / df["Num_Sent"]
    elif all(c in df.columns for c in ["Num_Fail_Other", "Num_Delay_Excd", "Num_Reliable"]):
        df["Reliability"] = df["Num_Reliable"] / (
            df["Num_Reliable"] + df["Num_Delay_Excd"] + df["Num_Fail_Other"]
        )
    else:
        raise ValueError("Required columns for Reliability not found.")
    return df

def combo_subset(df: pd.DataFrame, usi: float, bitrate: float, height: float) -> pd.DataFrame:
    return df.loc[(df["UAV_Sending_Interval"].to_numpy(dtype=float) == usi) & 
                  (df["Bitrate"].to_numpy(dtype=float) == bitrate) & (df["Height"].to_numpy(dtype=float) == height) &
                  (df["Height"].to_numpy(dtype=float) == height)]

all_cliff_dists = []
for link in LINKS:
    train_dataset_file_path = (
        f"/media/research-student/DataDrive/FANET_Dataset/"
        f"Dataset_NP100000_DJISpark/refined_train_dataset_processed/{link}_Reliability.csv"
    )
    train_df = add_reliability(pd.read_csv(train_dataset_file_path))
    cliff_dists = []
    for usi in USI:
        for bitrate in BITRATE:
            for height in TRAIN_HEIGHT:
                subset = combo_subset(train_df, usi, bitrate, height)
                cliff_hdist = find_dip_after_cliff_hdist(subset, threshold=THRESHOLD)
                if cliff_hdist > 0:
                    cliff_hdist = int(cliff_hdist)

                    # Refine cliff distance with hdist resolution of 1 m, with 10 points before the "dip after cliff" point.
                    # for d in range(cliff_hdist-9, cliff_hdist, 1):
                    #     cliff_dists.append({"USI": usi, "BitRate": bitrate, "MCS": mcs_bitrate_map[bitrate], "Height": height, "H_Dist": d})

                    # Refine cliff distance with hdist resolution of 0.1 m, with 10 points before the "dip after cliff" point. NOTE: Assumes cliff point can be found with resolution of 1 m for hdist.
                    for d in np.linspace(cliff_hdist-0.9, cliff_hdist-0.1, 9):
                        cliff_dists.append({"USI": usi, "BitRate": bitrate, "MCS": mcs_bitrate_map[bitrate], "Height": height, "H_Dist": d})
                        
    all_cliff_dists += cliff_dists

cliff_dist_df = pd.DataFrame(all_cliff_dists)
cliff_dist_df = cliff_dist_df.drop_duplicates()
print(len(cliff_dist_df))

2763


## Find Cliff Dist for Testing Dataset

In [2]:
'''
The cliff distance for each link may be different. Let's refine data points for all links.
Get the cliff distances for each link for each combination of MCS, USI, Height. Concatenate them and remove duplicated points.
'''

LINKS = ["Downlink", "Uplink", "Video"]
USI = [100]
BITRATE = [6.5, 13, 19.5, 26, 39, 52, 58.5, 65]  # treat as MCS
TEST_HEIGHT = [68, 82, 98, 112, 128, 142, 158, 172, 188, 202, 218, 232, 248, 262, 278, 292]
THRESHOLD = 0.99
mcs_bitrate_map = {6.5: "BPSK", 13: "QPSK", 19.5: "QPSK", 26: "QAM-16", 39: "QAM-16", 52: "QAM-64", 58.5: "QAM-64", 65: "QAM-64"}

def add_reliability(df: pd.DataFrame) -> pd.DataFrame:
    if "Num_Sent" in df.columns and "Num_Reliable" in df.columns:
        df["Reliability"] = df["Num_Reliable"] / df["Num_Sent"]
    elif all(c in df.columns for c in ["Num_Fail_Other", "Num_Delay_Excd", "Num_Reliable"]):
        df["Reliability"] = df["Num_Reliable"] / (
            df["Num_Reliable"] + df["Num_Delay_Excd"] + df["Num_Fail_Other"]
        )
    else:
        raise ValueError("Required columns for Reliability not found.")
    return df

def combo_subset(df: pd.DataFrame, usi: float, bitrate: float, height: float) -> pd.DataFrame:
    return df.loc[(df["UAV_Sending_Interval"].to_numpy(dtype=float) == usi) & 
                  (df["Bitrate"].to_numpy(dtype=float) == bitrate) & (df["Height"].to_numpy(dtype=float) == height) &
                  (df["Height"].to_numpy(dtype=float) == height)]

all_cliff_dists = []
for link in LINKS:
    train_dataset_file_path = (
        f"/media/research-student/DataDrive/FANET_Dataset/"
        f"Dataset_NP100000_DJISpark/refined_test_dataset_processed/{link}_Reliability.csv"
    )
    train_df = add_reliability(pd.read_csv(train_dataset_file_path))
    cliff_dists = []
    for usi in USI:
        for bitrate in BITRATE:
            for height in TEST_HEIGHT:
                subset = combo_subset(train_df, usi, bitrate, height)
                cliff_hdist = find_dip_after_cliff_hdist(subset, threshold=THRESHOLD)
                if cliff_hdist > 0:
                    cliff_hdist = int(cliff_hdist)

                    # Refine cliff distance with hdist resolution of 1 m, with 10 points before the "dip after cliff" point.
                    # for d in range(cliff_hdist-9, cliff_hdist, 1):
                    #     cliff_dists.append({"USI": usi, "BitRate": bitrate, "MCS": mcs_bitrate_map[bitrate], "Height": height, "H_Dist": d})

                    # Refine cliff distance with hdist resolution of 0.1 m, with 10 points before the "dip after cliff" point. NOTE: Assumes cliff point can be found with resolution of 1 m for hdist.
                    for d in np.linspace(cliff_hdist-0.9, cliff_hdist-0.1, 9):
                        cliff_dists.append({"USI": usi, "BitRate": bitrate, "MCS": mcs_bitrate_map[bitrate], "Height": height, "H_Dist": d})

    all_cliff_dists += cliff_dists

cliff_dist_df = pd.DataFrame(all_cliff_dists)
cliff_dist_df = cliff_dist_df.drop_duplicates()
print(len(cliff_dist_df))

2646


## Print values for .ini file

In [3]:
def to_ini_wrapped_list(values, per_line=40):
    def fmt(v):
        # Keeps integers clean (e.g., 60 not 60.0) and preserves decimals like 66.7
        if isinstance(v, float) and v.is_integer():
            return str(int(v))
        return f"{v:g}" if isinstance(v, float) else str(v)

    vals = [fmt(v) for v in values]
    lines = []
    for i in range(0, len(vals), per_line):
        chunk = ", ".join(vals[i:i + per_line])
        if i + per_line < len(vals):
            lines.append(chunk + ", \\")
        else:
            lines.append(chunk)
    return "\n".join(lines)

# If you want only H_Dist values
values = cliff_dist_df["H_Dist"].astype(float).tolist()

print("Horizontal Distances for Cliff Points:")
print(to_ini_wrapped_list(values, per_line=40))

print("Heights")
print(to_ini_wrapped_list(cliff_dist_df["Height"], per_line=40))

print("MCS")
print(to_ini_wrapped_list([f'"{v}"' for v in cliff_dist_df["MCS"]], per_line=40))

print("BitRate")
print(to_ini_wrapped_list(cliff_dist_df["BitRate"], per_line=40))

Horizontal Distances for Cliff Points:
278.1, 278.2, 278.3, 278.4, 278.5, 278.6, 278.7, 278.8, 278.9, 305.1, 305.2, 305.3, 305.4, 305.5, 305.6, 305.7, 305.8, 305.9, 333.1, 333.2, 333.3, 333.4, 333.5, 333.6, 333.7, 333.8, 333.9, 353.1, 353.2, 353.3, 353.4, 353.5, 353.6, 353.7, 353.8, 353.9, 374.1, 374.2, 374.3, 374.4, \
374.5, 374.6, 374.7, 374.8, 374.9, 390.1, 390.2, 390.3, 390.4, 390.5, 390.6, 390.7, 390.8, 390.9, 406.1, 406.2, 406.3, 406.4, 406.5, 406.6, 406.7, 406.8, 406.9, 420.1, 420.2, 420.3, 420.4, 420.5, 420.6, 420.7, 420.8, 420.9, 432.1, 432.2, 432.3, 432.4, 432.5, 432.6, 432.7, 432.8, \
432.9, 442.1, 442.2, 442.3, 442.4, 442.5, 442.6, 442.7, 442.8, 442.9, 453.1, 453.2, 453.3, 453.4, 453.5, 453.6, 453.7, 453.8, 453.9, 461.1, 461.2, 461.3, 461.4, 461.5, 461.6, 461.7, 461.8, 461.9, 467.1, 467.2, 467.3, 467.4, 467.5, 467.6, 467.7, 467.8, 467.9, 475.1, 475.2, 475.3, \
475.4, 475.5, 475.6, 475.7, 475.8, 475.9, 480.1, 480.2, 480.3, 480.4, 480.5, 480.6, 480.7, 480.8, 480.9, 483.1, 483